<a href="https://colab.research.google.com/github/marikhakhishvili/Facial-Expression-Recognition-Challenge/blob/main/model5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install kaggle wandb onnx -Uq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 23.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
! mkdir ~/.kaggle

In [5]:
!cp /content/drive/MyDrive/cs231n/assignments/assignment4/kaggle.json ~/.kaggle/kaggle.json

In [6]:
! chmod 600 ~/.kaggle/kaggle.json

download competition data

In [7]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge

100% 285M/285M [00:01<00:00, 174MB/s]



In [8]:
! unzip challenges-in-representation-learning-facial-expression-recognition-challenge

Archive:  challenges-in-representation-learning-facial-expression-recognition-challenge.zip
  inflating: example_submission.csv  
  inflating: fer2013.tar.gz          
  inflating: icml_face_data.csv      
  inflating: test.csv                
  inflating: train.csv               


In [9]:
import wandb
!wandb login --relogin

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Preprocessing

In [10]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

df = pd.read_csv('./train.csv')


def pixels_to_image_array(pixels_str):
    """
    Converts a space-separated pixel string into a normalized
    1 x 48 x 48 NumPy array suitable for PyTorch.
    """

    # Convert string to NumPy array of floats
    pixels = np.array(pixels_str.split(), dtype=np.float32)

    # Reshape into a 48x48 image
    image = pixels.reshape(48, 48)

    # Normalize pixel values from [0, 255] to [0, 1]
    image = image / 255.0

    # Add channel dimension: (48, 48) -> (1, 48, 48)
    image = np.expand_dims(image, axis=0)

    return image

# Apply preprocessing to every image
df["pixels"] = df["pixels"].apply(pixels_to_image_array)

# Check the shape of the first image
print(df["pixels"].iloc[0].shape)
# Expected output: (1, 48, 48)

(1, 48, 48)


In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np

X = np.stack(df["pixels"].values)   # shape: (N, 1, 48, 48)
y = df["emotion"].values

# First, split data into training+validation set (85%) and a separate local test set (15%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

# Then, split the training+validation set (85% of total) into actual training (70% of total) and validation sets (15% of total)
# This means the validation set will be 15% / 85% of X_train_val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(0.15 / 0.85), random_state=42, stratify=y_train_val
)

print(f"Original data size: {len(X)}")
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Local Test set size: {len(X_test)}")

Original data size: 28709
Training set size: 20095
Validation set size: 4307
Local Test set size: 4307


In [12]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.long)

X_test = torch.tensor(X_test, dtype=torch.float32) # New: Local Test set features
y_test = torch.tensor(y_test, dtype=torch.long)   # New: Local Test set labels

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
local_test_dataset = TensorDataset(X_test, y_test) # New: Local Test dataset

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
local_test_loader = DataLoader(local_test_dataset, batch_size=64, shuffle=False) # New: Local Test DataLoader

Model

In [13]:
import torchvision.models as models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=None)
criterion = nn.CrossEntropyLoss()

In [13]:
def train_one_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0

    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)

        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return total_loss / len(loader), correct / total

In [15]:
model.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

In [16]:
model.fc = nn.Linear(model.fc.in_features, 7)

In [17]:
model = model.to(device)

Training

In [18]:
wandb.init(
    project="fer2013-assignment",
    name="model5_resnet18",
    config={
        "model": "ResNet18",
        "lr": 3e-4,
        "batch_size": 64,
        "epochs": 25,
        "optimizer": "Adam",
        "weight_decay": 1e-4
    }
)

config = wandb.config

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mkhak23 (mkhak23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [19]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.lr,
    weight_decay=config.weight_decay
)

In [20]:
best_val_loss = float("inf")
patience = 5
epochs_without_improvement = 0

for epoch in range(config.epochs):

    train_loss, train_acc = train_one_epoch(model, train_loader)
    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1}/{config.epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(model.state_dict(), "best_resnet.pt")

    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break
model.load_state_dict(torch.load("best_resnet.pt"))


Epoch 1/25
Train Loss: 1.5791, Train Acc: 0.3840
Val Loss:   1.5515, Val Acc:   0.4156
Epoch 2/25
Train Loss: 1.3215, Train Acc: 0.4992
Val Loss:   1.4383, Val Acc:   0.4539
Epoch 3/25
Train Loss: 1.1198, Train Acc: 0.5819
Val Loss:   1.3655, Val Acc:   0.4922
Epoch 4/25
Train Loss: 0.9013, Train Acc: 0.6730
Val Loss:   1.4835, Val Acc:   0.4713
Epoch 5/25
Train Loss: 0.6636, Train Acc: 0.7622
Val Loss:   1.5950, Val Acc:   0.4890
Epoch 6/25
Train Loss: 0.4706, Train Acc: 0.8316
Val Loss:   1.8249, Val Acc:   0.4897
Epoch 7/25
Train Loss: 0.3209, Train Acc: 0.8897
Val Loss:   2.0490, Val Acc:   0.4999
Epoch 8/25
Train Loss: 0.2442, Train Acc: 0.9178
Val Loss:   2.1822, Val Acc:   0.4806
Early stopping triggered at epoch 8


<All keys matched successfully>

In [21]:
wandb.finish()

epoch,▁▂▃▄▅▆▇█
train_acc,▁▃▄▅▆▇██
train_loss,█▇▆▄▃▂▁▁
val_acc,▁▄▇▆▇▇█▆
val_loss,▃▂▁▂▃▅▇█
epoch,8
train_acc,0.91779
train_loss,0.24419
val_acc,0.48061
val_loss,2.18218


Training with dropout and augmentation

In [14]:
import torchvision.transforms as transforms

In [15]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),

    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [16]:
val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [17]:
class FERDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]   # (48,48)
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label

In [18]:
train_dataset1 = FERDataset(X_train, y_train, transform=train_transform)
val_dataset1 = FERDataset(X_val, y_val, transform=val_transform)
local_test_dataset_augmented = FERDataset(X_test, y_test, transform=val_transform) # New augmented test dataset

In [19]:
train_loader1 = torch.utils.data.DataLoader(
    train_dataset1,
    batch_size=64,
    shuffle=True
)

val_loader1 = torch.utils.data.DataLoader(
    val_dataset1,
    batch_size=64,
    shuffle=False
)

local_test_loader_augmented = torch.utils.data.DataLoader(
    local_test_dataset_augmented,
    batch_size=64,
    shuffle=False
) # New augmented test data loader

In [52]:
import torch.nn as nn

# Store the in_features attribute from the current model.fc layer.
# This ensures that even if the cell is re-run, the correct
# input features for the first Linear layer are used.
# For ResNet18, the default in_features for the final layer is 512.
fc_in_features = 512

model.fc = nn.Sequential(
    nn.Linear(fc_in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256, 7)
)

In [53]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [54]:
model = model.to(device)

In [55]:
wandb.finish() # Ensure any previous run is gracefully finished

wandb.init(
    project="fer2013-assignment",
    name="model_resnet18_augmented", # New run name
    config={
        "model": "ResNet18_Augmented",
        "lr": 3e-4,
        "batch_size": 64,
        "epochs": 25,
        "optimizer": "Adam",
        "weight_decay": 1e-4,
        "augmentation": "RandomHorizontalFlip, RandomRotation",
        "dropout": 0.5
    }
)

config = wandb.config

# Redefine the optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config.lr,
    weight_decay=config.weight_decay
)

# Retrain with augmented data and dropout
best_val_loss = float("inf")
epochs_without_improvement = 0
patience = 5

for epoch in range(config.epochs):

    train_loss, train_acc = train_one_epoch(model, train_loader1) # Use train_loader1
    val_loss, val_acc = evaluate(model, val_loader1)     # Use val_loader1

    print(f"Epoch {epoch+1}/{config.epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model.state_dict(), "best_resnet_augmented.pt")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

model.load_state_dict(torch.load("best_resnet_augmented.pt"))

# Evaluate on the local test set using the new augmented test loader
test_loss, test_acc = evaluate(model, local_test_loader_augmented)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

wandb.log({
    "test_loss": test_loss,
    "test_acc": test_acc
})
wandb.finish()

Epoch 1/25
Train Loss: 1.0795, Train Acc: 0.6100
Val Loss:   1.1880, Val Acc:   0.5531
Epoch 2/25
Train Loss: 1.0064, Train Acc: 0.6377
Val Loss:   1.2022, Val Acc:   0.5575
Epoch 3/25
Train Loss: 0.9647, Train Acc: 0.6535
Val Loss:   1.1944, Val Acc:   0.5668
Epoch 4/25
Train Loss: 0.9268, Train Acc: 0.6683
Val Loss:   1.2449, Val Acc:   0.5421
Epoch 5/25
Train Loss: 0.8837, Train Acc: 0.6793
Val Loss:   1.2901, Val Acc:   0.5526
Epoch 6/25
Train Loss: 0.8491, Train Acc: 0.6956
Val Loss:   1.2538, Val Acc:   0.5528
Early stopping triggered at epoch 6
Test Loss: 1.1815, Test Acc: 0.5579


epoch,▁▂▄▅▇█
test_acc,▁
test_loss,▁
train_acc,▁▃▅▆▇█
train_loss,█▆▅▃▂▁
val_acc,▄▅█▁▄▄
val_loss,▁▂▁▅█▆
epoch,6
test_acc,0.55793
test_loss,1.18155
train_acc,0.69565


trying other hyperparameters

In [20]:
import torchvision.models as models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model1 = models.resnet18(weights=None)
criterion = nn.CrossEntropyLoss()

In [21]:
model1.conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

In [22]:
import torch.nn as nn

# Store the in_features attribute from the current model.fc layer.
# This ensures that even if the cell is re-run, the correct
# input features for the first Linear layer are used.
# For ResNet18, the default in_features for the final layer is 512.
fc_in_features = 512

model1.fc = nn.Sequential(
    nn.Linear(fc_in_features, 256),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, 7)
)

In [25]:
model1 = model1.to(device)

In [26]:
wandb.init(
    project="fer2013-assignment",
    name="model_resnet18_augmented_dropout0.3", # New run name
    config={
        "model": "ResNet18_Augmented",
        "lr": 1e-4,
        "batch_size": 64,
        "epochs": 25,
        "optimizer": "Adam",
        "weight_decay": 5e-4,
        "augmentation": "RandomHorizontalFlip, RandomRotation",
        "dropout": 0.3
    }
)

config = wandb.config

optimizer = torch.optim.Adam(
    model1.parameters(),
    lr=config.lr,
    weight_decay=config.weight_decay
)

# Retrain with augmented data and dropout
best_val_loss = float("inf")
epochs_without_improvement = 0
patience = 5

for epoch in range(config.epochs):

    train_loss, train_acc = train_one_epoch(model1, train_loader1) # Use train_loader1
    val_loss, val_acc = evaluate(model1, val_loader1)     # Use val_loader1

    print(f"Epoch {epoch+1}/{config.epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        torch.save(model1.state_dict(), "best_resnet_augmented.pt")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

model1.load_state_dict(torch.load("best_resnet_augmented.pt"))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: mkhak23 (mkhak23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/25
Train Loss: 1.6569, Train Acc: 0.3394
Val Loss:   1.5051, Val Acc:   0.4161
Epoch 2/25
Train Loss: 1.4976, Train Acc: 0.4213
Val Loss:   1.4314, Val Acc:   0.4516
Epoch 3/25
Train Loss: 1.4155, Train Acc: 0.4576
Val Loss:   1.4010, Val Acc:   0.4623
Epoch 4/25
Train Loss: 1.3554, Train Acc: 0.4884
Val Loss:   1.3305, Val Acc:   0.4876
Epoch 5/25
Train Loss: 1.3028, Train Acc: 0.5078
Val Loss:   1.3070, Val Acc:   0.4980
Epoch 6/25
Train Loss: 1.2582, Train Acc: 0.5266
Val Loss:   1.3054, Val Acc:   0.5017
Epoch 7/25
Train Loss: 1.2076, Train Acc: 0.5467
Val Loss:   1.2867, Val Acc:   0.5157
Epoch 8/25
Train Loss: 1.1668, Train Acc: 0.5650
Val Loss:   1.2971, Val Acc:   0.5134
Epoch 9/25
Train Loss: 1.1228, Train Acc: 0.5817
Val Loss:   1.2669, Val Acc:   0.5226
Epoch 10/25
Train Loss: 1.0816, Train Acc: 0.5992
Val Loss:   1.2696, Val Acc:   0.5215
Epoch 11/25
Train Loss: 1.0402, Train Acc: 0.6151
Val Loss:   1.3064, Val Acc:   0.5261
Epoch 12/25
Train Loss: 1.0062, Train Acc

<All keys matched successfully>

In [28]:
test_loss, test_acc = evaluate(model1, local_test_loader_augmented)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

wandb.log({
    "test_loss": test_loss,
    "test_acc": test_acc
})
wandb.finish()

Test Loss: 1.2631, Test Acc: 0.5226


epoch,▁▂▂▃▃▄▄▅▅▆▆▇▇█
test_acc,▁
test_loss,▁
train_acc,▁▃▄▄▅▅▅▆▆▇▇▇██
train_loss,█▆▆▅▅▄▄▃▃▃▂▂▁▁
val_acc,▁▃▄▅▆▆▇▇▇▇▇███
val_loss,█▆▅▃▂▂▂▂▁▁▂▂▃▂
epoch,14
test_acc,0.52264
test_loss,1.26309
train_acc,0.66609
